# Week 3 — Day 1: Google Colab + Hugging Face + Image Generation

## My AI Engineering Notes

### What I am learning
- How Google Colab gives me temporary cloud compute.
- How to connect Colab to a GPU runtime such as a T4.
- How Hugging Face Hub is used to access models.
- How to keep my Hugging Face token private with Colab Secrets.
- How Diffusers can run image-generation models in Colab.
- Why model, library, dataset, and runtime versions must be compatible.

> **My mental model:** Hugging Face provides the models and tools; Colab provides the temporary compute to run them.

## 1. Google Colab — My Understanding

Google Colab is a cloud-hosted Jupyter Notebook environment. Instead of depending completely on my laptop's hardware, I can run Python on a remote machine.

### Why I use Colab for AI
1. I can access a GPU such as a T4 on supported runtimes.
2. I can share notebooks easily.
3. Reproduction is easier because the notebook contains the workflow.
4. It is useful when my local hardware is not enough.

### Important limitation
Colab runtimes are temporary. A runtime can reset, disconnect, or change hardware, so I should not treat the runtime disk as permanent storage.

## 2. My Colab Startup Checklist

- Connect to a hosted GPU runtime.
- Check the available GPU.
- Check GPU memory before loading a large model.
- Install the required package versions when reproducing an older tutorial.
- Keep important files outside the temporary runtime.

The lesson specifically checks for a **Tesla T4** and the displayed runtime has about **15 GB GPU memory**.

In [ ]:
# Check my NVIDIA GPU
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)

if 'failed' in gpu_info.lower():
    print('No NVIDIA GPU detected.')
else:
    print(gpu_info)
    if 'Tesla T4' in gpu_info:
        print('\n✅ T4 GPU detected')
    else:
        print('\nℹ️ A GPU is available, but it is not a T4.')

## 3. Why Package Versions Matter

Hugging Face libraries change quickly. The original lesson pins:

- `transformers==4.56.2`
- `diffusers==0.32.2`

The important lesson for me is: when reproducing an older tutorial, use its tested versions instead of blindly upgrading everything.

In [ ]:
# Only use this cell when I want to reproduce the tutorial's environment.
!pip install -q --upgrade transformers==4.56.2 diffusers==0.32.2

## 4. Hugging Face — What It Means to Me

For this lesson I mainly use:

- **Hugging Face Hub** → model access.
- **Transformers** → transformer-based models.
- **Diffusers** → diffusion/image-generation models.
- **Datasets** → supported datasets.
- **huggingface_hub** → authentication and Hub interaction.

```text
Hugging Face Hub
      ↓
   Model files
      ↓
   Colab GPU
      ↓
 Python library
      ↓
    Output
```

## 5. Hugging Face Token — Secure Method

I should **never hard-code my Hugging Face token** into a notebook that I plan to share.

### Colab method
1. Create/access my Hugging Face account.
2. Create an access token with the permissions required for the task.
3. Open Colab **Secrets** (key icon).
4. Create a secret named `HF_TOKEN`.
5. Turn notebook access on.
6. Read the secret from Python.

This keeps the actual token out of my notebook source.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

if not hf_token:
    raise ValueError('HF_TOKEN was not found in Colab Secrets.')

login(hf_token, add_to_git_credential=True)
print('✅ Hugging Face authentication completed.')

## 6. First Image Generator — SDXL Turbo

The lesson uses `stabilityai/sdxl-turbo` through Diffusers.

The important concept is that the model is loaded into the Colab runtime and the GPU performs the generation.

In [ ]:
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sdxl-turbo',
    torch_dtype=torch.float16,
    variant='fp16'
)
pipe.to('cuda')

prompt = 'A class of AI engineering students building an intelligent robot in a vibrant pop-art style'

image = pipe(
    prompt=prompt,
    num_inference_steps=4,
    guidance_scale=0.0
).images[0]

display(image)

## 7. What Just Happened?

```text
My text prompt
      ↓
SDXL Turbo pipeline
      ↓
GPU inference
      ↓
Generated image
```

### Key terms
- **Prompt** → instructions describing the image.
- **Pipeline** → a ready-to-use workflow containing model components.
- **Inference steps** → denoising computation.
- **CUDA** → lets PyTorch/Diffusers use the NVIDIA GPU.

## 8. Standard SDXL Base Model

The next experiment uses `stabilityai/stable-diffusion-xl-base-1.0`.

Compared with the quick Turbo experiment, this is a heavier workflow and therefore needs more computation.

In [ ]:
from diffusers import DiffusionPipeline
import torch
from IPython.display import display

pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    use_safetensors=True
)
pipe.to('cuda')

prompt = 'A group of data scientists learning AI engineering in a high-energy pop-art style'
image = pipe(prompt=prompt, num_inference_steps=30).images[0]
display(image)

## 9. SDXL Base + Refiner

The lesson also demonstrates a two-stage workflow:

```text
Prompt
  ↓
Base model
  ↓
Latent representation
  ↓
Refiner
  ↓
Final image
```

The lesson uses `n_steps = 40` and `high_noise_frac = 0.8`, so the base model handles the earlier part of denoising and the refiner handles the later part.

In [ ]:
from diffusers import DiffusionPipeline
import torch
from IPython.display import display

base = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True
)
base.to('cuda')

refiner = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-refiner-1.0',
    text_encoder_2=base.text_encoder_2,
    vae=base.vae,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True
)
refiner.to('cuda')

n_steps = 40
high_noise_frac = 0.8
prompt = 'A futuristic AI engineer designing an intelligent city in a vibrant high-energy pop-art style'

image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type='latent'
).images

image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image
).images[0]

display(image)

## 10. SpeechT5 Error — What I Learned

The SpeechT5 experiment is valuable because it demonstrates a dependency/API problem.

The model itself downloaded and loaded, but the old `matthijs/cmu-arctic-xvectors` dataset loader failed because current `datasets` versions no longer support that legacy Python dataset-loading script.

The important lesson:

> **A model loading successfully does not mean every dependency in the workflow is compatible.**

The failure occurred at the dataset-loading step, not at the SpeechT5 model-loading step.

I should identify the exact failing dependency instead of randomly reinstalling packages.

## 11. My Debugging Checklist

When a Hugging Face notebook fails:

1. Read the **last exception**.
2. Identify the exact line that failed.
3. Check installed versions.
4. Check whether the tutorial uses an old API.
5. Restart the runtime after changing core packages.
6. Re-run from the top after a runtime restart.
7. Never expose API/Hugging Face tokens in shared notebooks.

In [ ]:
import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 12. Runtime Rules I Should Remember

### Restart session
Useful after installing a different version of a package.

### Disconnect and delete runtime
This wipes the temporary runtime state. I should expect to reinstall packages and reload models.

### My rule
If a package says **restart the runtime**, I restart it before trusting the new version.

## 13. My Takeaways

I can now explain:

- What Google Colab is.
- Why a GPU runtime is useful for AI models.
- Why a T4 is useful for experimentation.
- What Hugging Face Hub provides.
- Why I should store `HF_TOKEN` in Colab Secrets.
- What Diffusers does.
- How a text prompt becomes an image through a diffusion pipeline.
- The basic difference between SDXL Turbo and the Base/Refiner workflow.
- Why package versions and runtime restarts matter.
- How to distinguish a model-loading problem from a dataset/dependency problem.

## 14. Practice Tasks — I Should Do These Myself

### Task 1 — GPU
Run `nvidia-smi` and record the GPU name and memory.

### Task 2 — Hugging Face
Authenticate using `HF_TOKEN` from Colab Secrets. Do not print the token.

### Task 3 — Image generation
Change the prompt to generate an image related to one of my AI/ML projects.

### Task 4 — Compare models
Generate one image with SDXL Turbo and one with SDXL Base. Record speed and visual differences.

### Task 5 — Debugging
Check the installed versions of `transformers`, `diffusers`, and `datasets` and explain why compatibility matters.

# Final Mental Model

```text
                HUGGING FACE HUB
                       │
             models / datasets
                       │
                       ▼
                  GOOGLE COLAB
                       │
                  T4 GPU runtime
                       │
          ┌────────────┴────────────┐
          ▼                         ▼
     Transformers               Diffusers
          │                         │
       NLP / TTS                Image AI
                                    │
                                    ▼
                              Generated image
```

**My main takeaway:** I am learning not just how to run a model, but how to connect the model, library, GPU runtime, authentication, and debugging workflow into one reproducible AI workflow.